# SPR addresses → cadastre id · walkthrough

Run this notebook **top to bottom, one cell at a time**. Each step saves its tables in `work/mapping.duckdb`, and the cells after it show what that step produced, so you can stop and check before going on.

**Rule:** an SPR row gets a `cadastre_id` only when exactly one cadastre address fits and the place, street, house and apartment all agree. Anything weaker becomes a suggestion or an issue with a named owner (CADASTRE / SPR / REVIEW).

Tools for investigating, usable after any step:

| function | what it shows |
|---|---|
| `explain(spr_id)` | everything the pipeline did with one SPR row |
| `search_cadastre('բաղրամյան', 'Արաբկիր')` | cadastre streets containing a text, optionally in one place |
| `streets_in('Արաբկիր')` | every cadastre street of a settlement / Yerevan district |
| `houses_on(street_uid)` | every cadastre address on one street |
| `sample('CAD_HOUSE_NOT_FOUND', 20)` | random rows of one issue category |
| `q("SELECT ...")` | any SQL on the working tables |

Input files go in `data/input/`; paths are in `config.py`.

In [1]:
import os, sys
sys.path.insert(0, os.getcwd())   # open this notebook from the project folder (the one with config.py)
assert os.path.exists('config.py'), 'Open the notebook from the project folder'

from steps import s01_load, s02_cadastre, s03_prepare_spr, s04_match_streets, s05_match_addresses, s06_export, s07_checks
from mapping.explore import q, explain, search_cadastre, streets_in, houses_on, sample

## Step 1 · Load the two files, untouched

- `raw_cadastre`, `raw_spr`: the files exactly as received.
- `spr`: `raw_spr` plus **`spr_id`**, an identity key 1..N in file order. Every output carries it.
- `match_type` / `cad_candidate_ids` (someone else's mapping) stay in the table but are never read.

In [2]:
s01_load.run()

[01_load] cadastre part cadastre_all_data_part1.xlsx: 487,929 rows


[01_load] cadastre part cadastre_all_data_part2.xlsx: 903,887 rows


[01_load] 5 cadastre rows were exact repeats (same id, same values); kept once


[01_load] cadastre rows: 1,391,811 (from 2 file(s))   spr rows: 614,083


In [3]:
q("""SELECT 'cadastre' AS file, count(*) AS rows FROM cadastre
     UNION ALL SELECT 'spr', count(*) FROM spr""")

,file,rows
0,cadastre,1391811
1,spr,614083


In [4]:
q("SELECT count(*) FROM spr where COALESCE(match_type, '') = 'AMBIGUOUS'")


,count_star()
0,87191


In [5]:
q('SELECT * FROM spr ORDER BY spr_id LIMIT 5')

,spr_id,match_type,cad_candidate_ids,region,community,settlement,street,house,house_type,apartment
0,1,AMBIGUOUS,"1000006,1252080,999599",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ՔԱՆԱՔԵՌ 12 Փ.,15,Տ,NaN
1,2,AMBIGUOUS,"1000006,1252080,999599",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ՔԱՆԱՔԵՌ 12 Փ. ԱՆՑՂ.,15,Տ,NaN
2,3,AMBIGUOUS,"10000,1228713,1470303",ՇԻՐԱԿ,ԱՇՈՑՔ,ԱՇՈՑՔ,3 ԹՂՄ. 4 Փ.,3,Շ,3
3,4,AMBIGUOUS,"1000014,1251663",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ԶԱՔԱՐԻԱ ՔԱՆԱՔԵՌՑՈՒ Փ.,151/1,Շ,38
4,5,AMBIGUOUS,"1000019,999195",ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,ԶԱՔԱՐԻԱ ՔԱՆԱՔԵՌՑՈՒ Փ.,151/1,Շ,11


**Check:** a cadastre file with exactly **1,048,575** rows hit Excel's sheet limit, so the export is probably cut off. Houses missing in later steps may be missing only from this file.

## Step 2 · Cadastre keys and cadastre data issues

- `cadastre_keys`: one row per cadastre id with cleaned geography, a `street_uid` and cleaned house / building numbers.
- `cadastre_streets`: one row per distinct street per settlement.
- `cadastre_issues`: problems in the cadastre itself, found without looking at SPR.

In [6]:
s02_cadastre.run()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[02_cadastre] cadastre streets: 38,357


[02_cadastre]    CAD_DUPLICATE_ADDRESS              225,538 ids


[02_cadastre]    CAD_SAME_ADDRESS_MULTI_OBJECT      119,617 ids


[02_cadastre]    CAD_NONSTANDARD_HOUSE_NUMBER        23,293 ids


[02_cadastre]    CAD_STREET_NAME_MISSING              9,352 ids


[02_cadastre]    CAD_LEADING_TRAILING_SPACES          4,600 ids


[02_cadastre]    CAD_STREET_SPELLING_VARIANTS            46 ids


In [7]:
q('SELECT * FROM cadastre_streets USING SAMPLE 10 ROWS (reservoir, 1)')

,street_uid,region_k,community_k,settlement_k,street_key,variant_key,street_name,street_type,sub_street_name,sub_street_type,region,community,settlement,n_addresses,is_spelling_variant
0,148,արագածոտն,աշտարակ,ագարակ,12|փողոց|1|փակուղի,12|փողոց|1|փակուղի,12-րդ,փողոց,1-ին,փակուղի,Արագածոտն,Աշտարակ,Ագարակ,9,False
1,6848,արարատ,մասիս,ղուկասավան,4|փողոց||,4|փողոց||,4-րդ,փողոց,NaN,NaN,Արարատ,Մասիս,Ղուկասավան,40,False
2,9608,արմավիր,խոյ,արշալույս,8|փողոց||,8|փողոց||,8-րդ,փողոց,NaN,NaN,Արմավիր,Խոյ,Արշալույս,15,False
3,17793,երեվան,երեվան,մալաթիա սեբաստիա,հ ծերենցի|փողոց||նրբանցք,ծերենց|փողոց||նրբանցք,Հ. Ծերենցի,փողոց,NaN,նրբանցք,Երևան,Երևան,Մալաթիա-Սեբաստիա,7,False
4,20606,լոռի,սպիտակ,սարալանջ,3|փողոց|1|նրբանցք,3|փողոց|1|նրբանցք,3-րդ,փողոց,1-ին,նրբանցք,Լոռի,Սպիտակ,Սարալանջ,9,False
5,25998,կոտայք,նոր հաճըն,արգել,8|փողոց||,8|փողոց||,8-րդ,փողոց,NaN,NaN,Կոտայք,Նոր Հաճըն,Արգել,12,False
6,26893,կոտայք,չարենցավան,բջնի,9|փողոց|4|նրբանցք,9|փողոց|4|նրբանցք,9-րդ,փողոց,4-րդ,նրբանցք,Կոտայք,Չարենցավան,Բջնի,4,False
7,32328,սյունիք,սիսիան,գորայք,ա սարուխանյան|փողոց||,սարուխանյան|փողոց||,Ա. Սարուխանյան,փողոց,NaN,NaN,Սյունիք,Սիսիան,Գորայք,24,False
8,34230,վայոց ձոր,եղեգնաձոր,եղեգնաձոր,սիրանեսի|փողոց|2|նրբանցք,սիրանես|փողոց|2|նրբանցք,Սիրանեսի,փողոց,2-րդ,նրբանցք,Վայոց ձոր,Եղեգնաձոր,Եղեգնաձոր,1,False
9,36054,տավուշ,դիլիջան,դիլիջան,ս օրջոնիկիձեի|փողոց|5|նրբանցք,օրջոնիկիձե|փողոց|5|նրբանցք,Ս. Օրջոնիկիձեի,փողոց,5-րդ,նրբանցք,Տավուշ,Դիլիջան,Դիլիջան,36,False


In [8]:
q('SELECT issue_code, count(*) AS cadastre_ids FROM cadastre_issues GROUP BY 1 ORDER BY 2 DESC')

,issue_code,cadastre_ids
0,CAD_DUPLICATE_ADDRESS,225538
1,CAD_SAME_ADDRESS_MULTI_OBJECT,119617
2,CAD_NONSTANDARD_HOUSE_NUMBER,23293
3,CAD_STREET_NAME_MISSING,9352
4,CAD_LEADING_TRAILING_SPACES,4600
5,CAD_STREET_SPELLING_VARIANTS,46


Examples: identical cadastre addresses with several ids.

In [9]:
q("""SELECT i.detail, c.region, c.settlement, c.street_name, c.street_type, c.house, c.house_type, c.building
     FROM cadastre_issues i JOIN cadastre c ON c.id = i.cadastre_id
     WHERE i.issue_code = 'CAD_DUPLICATE_ADDRESS' ORDER BY random() LIMIT 10""")

,detail,region,settlement,street_name,street_type,house,house_type,building
0,"same address and same type; ids: 1108798,1263687",Արագածոտն,Փարպի,Ղ. Փարպեցու,փողոց,51,հողամաս,NaN
1,"same address and same type; ids: 1232150,1467930",Գեղարքունիք,Գավառ,Զոհված Ազատամարտիկների,փողոց,2/58,ավտոտնակ,NaN
2,"same address and same type; ids: 1443460,669926",Վայոց ձոր,Եղեգնաձոր,Խնձորի այգի,թաղամաս,10/2,հողամաս,NaN
3,"same address and same type; ids: 1429965,242491",Կոտայք,Զովք,10-րդ,փողոց,17,հողամաս,NaN
4,"same address and same type; ids: 963355,963357",Երևան,Նոր Նորք,Ար. Միկոյան,փողոց,6,հողամաս,NaN
5,"same address and same type; ids: 1137836,1195667",Երևան,Աջափնյակ,Նորաշեն,թաղամաս,44,շենք,76
6,"same address and same type; ids: 927260,927270",Երևան,Նոր Նորք,8-րդ,միկրոշրջան,2,շենք,9
7,"same address and same type; ids: 1302258,1400257",Երևան,Կենտրոն,Արշակունյաց,պողոտա,"18/4 շենք, 4-րդ",մասնաշենք,153
8,"same address and same type; ids: 1118289,1118290",Լոռի,Հոբարձի,14-րդ,փողոց,17,հողամաս,NaN
9,"same address and same type; ids: 1216797,960137,960138",Երևան,Նորք-Մարաշ,Նորքի այգիների,փողոց,203,բնակելի տուն,NaN


## Step 3 · Put SPR geography on the cadastre's hierarchy

Colleagues' corrections in their order: `geo_alias` → `community_alias` → Yerevan (district moves from community to settlement) → Gyumri → as is.
Each row also gets the colleagues' `data_completeness` (% of the 6 address parts that are filled).

In [10]:
from steps import s03_prepare_spr
s03_prepare_spr.run()

[03_prepare_spr] rows: 614,083


[03_prepare_spr]    geo_status ok                             515,774


[03_prepare_spr]    geo_status foreign_or_unknown_region       96,297


[03_prepare_spr]    geo_status region_missing                   1,683


[03_prepare_spr]    geo_status community_not_in_cadastre          279


...

[03_prepare_spr]    geo_rule   as_is                          409,690


[03_prepare_spr]    geo_rule   yerevan                        156,624


[03_prepare_spr]    geo_rule   gyumri                          39,753


[03_prepare_spr]    geo_rule   geo_alias                        7,843


[03_prepare_spr]    geo_rule   community_alias                    173


In [11]:
q('SELECT geo_status, geo_note, count(*) AS rows FROM spr_prepared GROUP BY ALL ORDER BY rows DESC')

,geo_status,geo_note,rows
0,ok,NaN,505823
1,foreign_or_unknown_region,NaN,96297
2,ok,district_missing,9108
3,region_missing,NaN,1683
4,ok,settlement_missing,809
5,community_not_in_cadastre,NaN,279
6,settlement_not_in_cadastre,NaN,50
7,ok,community_missing,23
8,ok,district_not_in_cadastre,11


In [12]:
q('SELECT geo_rule, count(*) AS rows FROM spr_prepared GROUP BY 1 ORDER BY 2 DESC')

,geo_rule,rows
0,as_is,409690
1,yerevan,156624
2,gyumri,39753
3,geo_alias,7843
4,community_alias,173


Corrections in action (one example per rule):

In [13]:
q("""SELECT DISTINCT ON (p.geo_rule) p.geo_rule, s.region, s.community, s.settlement,
            p.community_k AS community_used, p.settlement_k AS settlement_used
     FROM spr_prepared p JOIN spr s USING (spr_id)""")

,geo_rule,region,community,settlement,community_used,settlement_used
0,gyumri,ՇԻՐԱԿ,ԳՅՈՒՄՐԻ,NaN,գյումրի,գյումրի
1,community_alias,ԿՈՏԱՅՔ,ՄԵՂՐԱՁՈՐ ԲԱՆԱՎԱՆ,NaN,մեղրաձոր,
2,yerevan,ԵՐԵՎԱՆ,ՇԵՆԳԱՎԻԹ,NaN,երեվան,շենգավիթ
3,as_is,էէ,էէէ,NaN,էէէ,
4,geo_alias,ԼՈՌԻ,ԼՈՐՈԻՏ,ԼՈՐՈԻՏ,թումանյան,լորուտ


Places that do not exist in the cadastre (with a hint when the settlement is found under another community):

In [14]:
q("""SELECT s.region, s.community, s.settlement, p.geo_status, p.geo_hint, count(*) AS rows
     FROM spr_prepared p JOIN spr s USING (spr_id)
     WHERE p.geo_status IN ('community_not_in_cadastre', 'settlement_not_in_cadastre')
     GROUP BY ALL ORDER BY rows DESC LIMIT 20""")

,region,community,settlement,geo_status,geo_hint,rows
0,ԿՈՏԱՅՔ,ՄԵՂՐԱՁՈՐ ԲԱՆԱՎԱՆ,NaN,community_not_in_cadastre,is a settlement of community: ծաղկաձոր,42
1,ԱՐԱՐԱՏ,ՎԵԴՈՒ Գ/Գ ԲԱՆԱՎԱՆ,NaN,community_not_in_cadastre,NaN,25
2,ԱՐԱԳԱԾՈՏՆ,ԱՄՐԵ ԹԱԶԱ,NaN,community_not_in_cadastre,NaN,17
3,ՇԻՐԱԿ,ԳՅՈՒՄՐԻ,ԱՐՏԵՆԻ,settlement_not_in_cadastre,NaN,15
4,ԱՐԱՐԱՏ,ԶԻՆԱՎԱՆ 5,NaN,community_not_in_cadastre,NaN,14
5,ՇԻՐԱԿ,ԳՅՈՒԼԼԻԲՈՒԼԱՂ,NaN,community_not_in_cadastre,NaN,14
6,ՍՅՈՒՆԻՔ,ՈՒԽՏԱՎԱՅՐ,NaN,community_not_in_cadastre,NaN,13
7,ՍՅՈՒՆԻՔ,ԽԱԼԱՋ,NaN,community_not_in_cadastre,NaN,13
8,ՍՅՈՒՆԻՔ,ՆԺԴԵՀ,ՄՈՒՐԽՈՒԶ,community_not_in_cadastre,is a settlement of community: սիսիան,11
9,ՇԻՐԱԿ,ՋԱՋՈՒՌ ԿԱՅԱՐԱՆ,NaN,community_not_in_cadastre,NaN,7


In [15]:
q('SELECT data_completeness, count(*) AS rows FROM spr_prepared GROUP BY 1 ORDER BY 1 DESC')

,data_completeness,rows
0,100.00,113721
1,83.33,341747
2,66.67,92957
3,50.00,39067
4,33.33,24804
5,16.67,1787


## Step 4 · Find the cadastre street

For each search area (settlement first), each rule in order, each reading of the text; the **first hit decides**.

| # | rule | allows | confidence |
|---|---|---|---|
| 1 | exact | identical cleaned name | high |
| 2 | alias | colleagues' renames (Գյումրի, Կապան) | high |
| 3 | double_letter | notebook typo list, doubled letters | high |
| 4 | word_form | genitive endings (`ԳԱՌՆԻԻ` = `Գառնու`) | high |
| 5 | extra_words | SPR words ⊂ cadastre words (`ԱՆԴՐԱՆԻԿԻ` = `Զորավար Անդրանիկի`) | medium |
| 6 | initial_for_name | `Ք.` = `ՔԸՐՔ`, or one side omits the initial | medium |
| 7 | fuzzy | letters only, score ≥ 90 and 4 clear of the next street | medium |

Numbers (street number, block code, lane) are never fuzzy: they must be identical and in the same order.

Try the street parser on any text first:

In [16]:
from mapping.street_parser import parse_spr_street
for text in ['ՍՈՍԵՅԻ Փ. 1 ՆՐԲ.', 'Ա2 ԹՂՄ. 9 Փ.', 'ԴՐՈՅԻ 4 ՆՐԲ.', 'ՔԸՐՔ ՔԸՐՔՈՐՅԱՆ Փ.', 'ՔԱՆԱՔԵՌ 12 Փ. ԱՆՑՂ.']:
    print(f'{text:22}', [(r, st.text(), leftover) for r, st, leftover, _ in parse_spr_street(text)])

ՍՈՍԵՅԻ Փ. 1 ՆՐԲ.       [('as_written', 'սոսեյի | փողոց | 1 | նրբանցք', 'none')]
Ա2 ԹՂՄ. 9 Փ.           [('neighbourhood_street', 'ա2 թաղամասի 9 | փողոց', 'none')]
ԴՐՈՅԻ 4 ՆՐԲ.           [('as_written', 'դրոյի 4 | նրբանցք', 'none'), ('lane_of_street', 'դրոյի | 4 | նրբանցք', 'none')]
ՔԸՐՔ ՔԸՐՔՈՐՅԱՆ Փ.      [('as_written', 'քըրք քըրքորյան | փողոց', 'none')]
ՔԱՆԱՔԵՌ 12 Փ. ԱՆՑՂ.    [('as_written', 'քանաքեռ 12 | փողոց | անցուղի', 'none')]


In [17]:
s04_match_streets.run()

[04_match_streets] distinct (scope, street) pairs to match: 48,812


[04_match_streets]    20,000 done


[04_match_streets]    40,000 done


[04_match_streets] rows matched at street level: 515,774


[04_match_streets]    found                  407,929


...

[04_match_streets]    type_mismatch           16,016


[04_match_streets]    no_readings             13,210


[04_match_streets]    suggested                8,160


[04_match_streets]    ambiguous                2,994


[04_match_streets]    no_street_in_spr           726


In [18]:
q('SELECT street_status, count(*) AS rows FROM spr_street_match GROUP BY 1 ORDER BY 2 DESC')

,street_status,rows
0,found,407929
1,not_found,66739
2,type_mismatch,16016
3,no_readings,13210
4,suggested,8160
5,ambiguous,2994
6,no_street_in_spr,726


In [19]:
q("""SELECT match_method, text_reading, type_check, count(*) AS rows FROM spr_street_match
     WHERE street_status = 'found' GROUP BY ALL ORDER BY rows DESC""")

,match_method,text_reading,type_check,rows
0,exact,as_written,same type,206635
1,initial_for_name,as_written,same type,93699
2,extra_words,as_written,same type,43727
3,word_form,as_written,same type,13137
4,alias,as_written,set by alias,12384
5,exact,as_written,issue has no type,7054
6,exact,neighbourhood_street,same type,6823
7,initial_for_name,lane_of_street,issue has no type,4119
8,exact,lane_of_street,issue has no type,4059
9,word_form,without_house_numbers,issue has no type,3051


SPR street next to the cadastre street it was matched to — change `METHOD` to check any rule:

In [20]:
METHOD = 'initial_for_name'   # exact, alias, double_letter, word_form, extra_words, initial_for_name, fuzzy
q(f"""SELECT s.settlement, s.community, s.street AS spr_street, m.text_reading,
            cs.street_name, cs.street_type, cs.sub_street_name, cs.sub_street_type, m.similarity_score
     FROM spr_street_match m JOIN spr s USING (spr_id) JOIN cadastre_streets cs ON cs.street_uid = m.street_uid
     WHERE m.street_status = 'found' AND m.match_method = '{METHOD}'
     ORDER BY random() LIMIT 20""")

,settlement,community,spr_street,text_reading,street_name,street_type,sub_street_name,sub_street_type,similarity_score
0,ՆՈՅԵՄԲԵՐՅԱՆ,ՆՈՅԵՄԲԵՐՅԱՆ,ԱԲՈՎՅԱՆ Փ.,as_written,Խ. Աբովյան,փողոց,None,None,NaN
1,NaN,ԱՋԱՓՆՅԱԿ,ԼՈՒԿԱՇԻՆԻ 1 Փ.,as_written,Ս. Լուկաշինի 1-ին,փողոց,None,None,NaN
2,ՎԱՐԴԱՇԵՆ,ԱՐՏԱՇԱՏ,ԽԱՉԱՏՈՒՐ ԱԲՈՎՅԱՆ Փ.,as_written,Խ. Աբովյան,փողոց,None,None,NaN
3,ԱՐՏԱՇԱՏ,ԱՐՏԱՇԱՏ,ԴՈՒՐՅԱՆ Փ.,as_written,Պ. Դուրյան,փողոց,None,None,NaN
4,ՎԱՆԱՁՈՐ,ՎԱՆԱՁՈՐ,ՀԱԿՈԲՅԱՆ Փ.,as_written,Հ. Հակոբյան,փողոց,None,None,NaN
5,ՎԱՅՔ,ՎԱՅՔ,ՇԱՀՈՒՄՅԱՆ Փ.,as_written,Ս. Շահումյան,փողոց,None,None,NaN
6,NaN,ԱՐԱԲԿԻՐ,ՎԱՂԱՐՇՅԱՆ Փ.,as_written,Վ. Վաղարշյան,փողոց,None,None,NaN
7,NaN,ԿԵՆՏՐՈՆ,ԹՈՒՄԱՆՅԱՆ Փ.,as_written,Հ. Թումանյան,փողոց,None,None,NaN
8,NaN,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,ՀԱՍՐԱԹՅԱՆ Փ.,as_written,Է. Հասրաթյան,փողոց,None,None,NaN
9,ՄԱՐԱԼԻԿ,ԱՆԻ,ԱԲՈՎՅԱՆ Փ.,as_written,Խ. Աբովյան,փողոց,None,None,NaN


Most frequent street texts that were **not found** (biggest impact first):

In [21]:
q("""SELECT s.region, s.community, s.settlement, s.street, m.parsed_street, count(*) AS rows
     FROM spr_street_match m JOIN spr s USING (spr_id) WHERE m.street_status = 'not_found'
     GROUP BY ALL ORDER BY rows DESC LIMIT 20""")

,region,community,settlement,street,parsed_street,rows
0,ՇԻՐԱԿ,ԱԽՈՒՐՅԱՆ,ԱԽՈՒՐՅԱՆ,ՇՐՋԿԵՆՏՐՈՆ,շրջկենտրոն,1152
1,ԵՐԵՎԱՆ,ՆՈՐ ՆՈՐՔ,NaN,ՋՐՎԵԺ ՄԱՅԱԿ,ջրվեժ մայակ,1131
2,ԿՈՏԱՅՔ,ՀՐԱԶԴԱՆ,ՀՐԱԶԴԱՆ,ԿԵՆՏՐՈՆ ՀՄ ՀՆԿՑ.,կենտրոն հմ,1073
3,ԵՐԵՎԱՆ,ԿԵՆՏՐՈՆ,NaN,ՊԱՊ ԹԱԳԱՎՈՐԻ Փ.,պապ թագավորի | փողոց,988
4,ԼՈՌԻ,ՎԱՆԱՁՈՐ,ՎԱՆԱՁՈՐ,ՏԱՐՈՆ-4 ՄԵՔ.,տարոն 4 մեք,980
5,ԵՐԵՎԱՆ,ՆՈՐ ՆՈՐՔ,NaN,ՋՐՎԵԺ ԲԱՆԱՎԱՆ,ջրվեժ բանավան,897
6,ԵՐԵՎԱՆ,ԱՐԱԲԿԻՐ,NaN,ՔԱՆԱՔԵՌ ՀԷԿ ԲԱՆԱՎԱՆ,քանաքեռ հէկ բանավան,718
7,ԵՐԵՎԱՆ,ԱՎԱՆ,NaN,ԱՎԱՆ ԱՌԻՆՋ,ավան առինջ,705
8,ԵՐԵՎԱՆ,ՇԵՆԳԱՎԻԹ,NaN,Ն. ՇԵՆԳԱՎԻԹ 11 Փ.,ն շենգավիթ 11 | փողոց,649
9,ԵՐԵՎԱՆ,ՔԱՆԱՔԵՌ-ԶԵՅԹՈՒՆ,NaN,Մ. ԱՎԵՏԻՍՅԱՆ 4 Փ.,մ ավետիսյան 4 | փողոց,625


Name found but type differs (`Փ.` vs `պողոտա` …) — not mapped:

In [22]:
q("""SELECT s.settlement, s.community, s.street, m.parsed_street, cs.street_name, cs.street_type, count(*) AS rows
     FROM spr_street_match m JOIN spr s USING (spr_id)
     JOIN cadastre_streets cs ON cs.street_uid = TRY_CAST(split_part(m.candidate_uids, ',', 1) AS BIGINT)
     WHERE m.street_status = 'type_mismatch' GROUP BY ALL ORDER BY rows DESC LIMIT 15""")

,settlement,community,street,parsed_street,street_name,street_type,rows
0,NaN,ՆՈՐ ՆՈՐՔ,ՆՈՐ ՆՈՐՔ 7 ԶՆԳՎ.,նոր նորք 7 | զանգված,7-րդ,միկրոշրջան,2321
1,NaN,ՆՈՐ ՆՈՐՔ,ՆՈՐ ՆՈՐՔ 8 ԶՆԳՎ.,նոր նորք 8 | զանգված,8-րդ,միկրոշրջան,1775
2,NaN,ԳՅՈՒՄՐԻ,Խ. ՀԱՅՐԻԿ ԽՃՂ.,խ հայրիկ | խճուղի,Խրիմյան Հայրիկի,փողոց,858
3,ԵՂՎԱՐԴ,ՆԱԻՐԻ,ԵՐԵՎԱՆՅԱՆ Փ.,երեվանյան | փողոց,Երևանյան,թաղամաս,625
4,ՋԵՐՄՈՒԿ,ՋԵՐՄՈՒԿ,ՁԱԽԱՓՆՅԱԿ Փ.,ձախափնյակ | փողոց,Ձախափնյակ,թաղամաս,577
5,NaN,ՆՈՐ ՆՈՐՔ,ԳՅՈՒԼԻՔԵԽՎՅԱՆ ՆՐԲ.,գյուլիքեխվյան | նրբանցք,Հ. Գյուլիքևխյան,փողոց,396
6,NaN,ՇԵՆԳԱՎԻԹ,ԵՂ. ԹԱԴԵՎՈՍՅԱՆ ՆՐԲ.,եղ թադեվոսյան | նրբանցք,Ե. Թադևոսյան,փողոց,375
7,NaN,ՆՈՐ ՆՈՐՔ,Դ. ՄԱԼՅԱՆ ՆՐԲ.,դ մալյան | նրբանցք,Դ. Մալյան,փողոց,314
8,NaN,ԱՎԱՆ,ԱՎԱՆ 7 Փ.,ավան 7 | փողոց,7-րդ,միկրոշրջան,313
9,ԱԼԱՎԵՐԴԻ,ԱԼԱՎԵՐԴԻ,ԲԱՂՐԱՄՅԱՆ Փ.,բաղրամյան | փողոց,Մարշալ Բաղրամյան,թաղամաս,300


## Step 5 · House, apartment, object type → cadastre id

House and apartment must be **identical** after cleaning; no SPR apartment means the cadastre row must have no building number either. If several ids remain, the SPR house type (`Տ`, `Շ`, `ՀՈՂ`) picks the matching object. A single id that is a different kind of object (garage for an apartment, land plot for a house) becomes a suggestion.

In [23]:
s05_match_addresses.run()

[05_match_addresses] rows with one street (found or suggested): 430,694


[05_match_addresses]    street suggested apartment_missing_in_spr                   128


[05_match_addresses]    street suggested apartment_not_found                      3,014


[05_match_addresses]    street suggested cadastre_duplicate                         879


[05_match_addresses]    street suggested house_has_no_units_in_cadastre             607


...

[05_match_addresses]    street found     house_not_found                        137,763


[05_match_addresses]    street found     mapped                                 116,643


[05_match_addresses]    street found     object_type_conflict                     1,428


[05_match_addresses]    street found     several_objects                          2,260


[05_match_addresses]    street found     street_text_number_conflicts_house       3,819


In [24]:
q("""SELECT street_confirmed, address_status, address_pick, object_type_check, count(*) AS rows
     FROM spr_address_match GROUP BY ALL ORDER BY street_confirmed DESC, rows DESC""")

,street_confirmed,address_status,address_pick,object_type_check,rows
0,True,house_not_found,NaN,NaN,137763
1,True,mapped,single,same,79673
2,True,apartment_not_found,NaN,NaN,78486
3,True,cadastre_duplicate,NaN,NaN,49663
4,True,mapped,by_object_type,same,27578
5,True,house_has_no_units_in_cadastre,NaN,NaN,13445
6,True,mapped,single,unknown,5835
7,True,street_text_number_conflicts_house,NaN,NaN,3819
8,True,apartment_missing_in_spr,NaN,NaN,2686
9,True,several_objects,NaN,NaN,2260


Street found, house missing in the cadastre — the houses that street *does* have:

In [25]:
ex = q("""SELECT a.spr_id, a.street_uid, a.house_k FROM spr_address_match a
          WHERE a.address_status = 'house_not_found' ORDER BY random() LIMIT 1""")
print(ex.to_string(index=False))
houses_on(int(ex.street_uid[0]))

 spr_id  street_uid house_k
 497704       24137      22


,cadastre_id,house,house_type,building,building_type
0,494120,10,բնակելի տուն,None,None
1,494501,12,բնակելի տուն,None,None
2,494121,12,բնակելի տուն,None,None
3,494502,13,բնակելի տուն,None,None
4,494503,16,բնակելի տուն,None,None
5,494504,18,բնակելի տուն,None,None
6,1312636,18,NaN,None,None
7,1224535,18/2,բնակելի տուն,None,None
8,1470462,18/2,հողամաս,None,None
9,494505,2,բնակելի տուն,None,None


## Step 6 · Final status, issue categories, output files

`match_status`: mapped · suggested · ambiguous · not_found · out_of_scope.
Only rows found **in the place the SPR row names** are mapped; a hit in another Yerevan district, or in a wider search because SPR gives no district / settlement, is a suggestion.

In [26]:
s06_export.run()

[06_export]    not_found                 329,278


[06_export]    out_of_scope               97,980


[06_export]    mapped         medium      83,779


[06_export]    ambiguous                  54,784


[06_export]    mapped         high        32,167


[06_export]    suggested                  16,095


[06_export] written to /Users/nareabgaryan/Desktop/ISAAPR1/spr_cadastre_mapping/output


In [27]:
q("""SELECT match_status, match_confidence, count(*) AS rows,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
     FROM spr_mapping_result GROUP BY ALL ORDER BY rows DESC""")

,match_status,match_confidence,rows,pct
0,not_found,NaN,329278,53.62
1,out_of_scope,NaN,97980,15.96
2,mapped,medium,83779,13.64
3,ambiguous,NaN,54784,8.92
4,mapped,high,32167,5.24
5,suggested,NaN,16095,2.62


In [28]:
q("""SELECT r.issue_category, i.owner, i.match_status, count(*) AS rows, i.meaning
     FROM spr_mapping_result r JOIN issue_ref i ON i.code = r.issue_category
     GROUP BY ALL ORDER BY rows DESC""")

,issue_category,owner,match_status,rows,meaning
0,CAD_HOUSE_NOT_FOUND,CADASTRE,not_found,137491,"Street exists in the cadastre, this house number does not"
1,SPR_FOREIGN_ADDRESS,SPR,out_of_scope,96297,"Region is not one of the 11 cadastre regions (another country, ԼՂՀ..."
2,CAD_APARTMENT_NOT_FOUND,CADASTRE,not_found,78437,"House exists with apartments, this apartment number does not"
3,STREET_NOT_FOUND,REVIEW,not_found,67749,"No cadastre street with this name in the place (old name, typo bey..."
4,CAD_DUPLICATE_ADDRESS,CADASTRE,ambiguous,49647,Several cadastre ids with identical address and object type
5,CAD_HOUSE_HAS_NO_UNITS,REVIEW,not_found,13423,SPR gives an apartment; the cadastre house has no apartments at all
6,SPR_STREET_TEXT_UNPARSED,SPR,not_found,13210,Street text holds parts that change the address and cannot be read...
7,STREET_TYPE_MISMATCH,REVIEW,not_found,9511,"Street name found, but SPR type (Փ., ՆՐԲ. ...) differs from the ca..."
8,STREET_STREETLESS_SUGGESTED,REVIEW,suggested,7675,SPR has no street; settlement has street-less cadastre addresses b...
9,ADDRESS_FOUND_ON_SUGGESTED_STREET,REVIEW,suggested,6149,"Street only suggested or type differs, but this exact house/apartm..."


In [29]:
q("""SELECT street_match_method, match_confidence, count(*) AS rows FROM spr_mapping_result
     WHERE match_status = 'mapped' GROUP BY ALL ORDER BY rows DESC""")

,street_match_method,match_confidence,rows
0,initial_for_name,medium,30978
1,extra_words,medium,29536
2,exact,medium,21395
3,exact,high,20145
4,alias,high,8256
5,word_form,high,2601
6,double_letter,high,1165
7,alias,medium,967
8,word_form,medium,764
9,double_letter,medium,112


## Step 7 · Integrity checks (the cell fails if any check fails)

In [30]:
s07_checks.run()

[07_checks] PASS  result has one row per SPR file row


[07_checks] PASS  spr_id is unique and runs 1..N


[07_checks] PASS  spr table equals the raw file (no value changed)


[07_checks] PASS  cadastre table equals the raw file


[07_checks] PASS  cadastre_id is set exactly for mapped rows


...

[07_checks] PASS  mapped Yerevan rows: same district (raw values)


[07_checks] PASS  mapped: found in the settlement / district the SPR names


[07_checks] PASS  mapped without a place in SPR: only Yerevan rows without district (nare_rules #6)


[07_checks] PASS  Yerevan rows with a district: no suggested id in another district


[07_checks] info: 1,929 cadastre ids are the target of more than one SPR row (the same address written differently in SPR)


## Investigate

Pick any row and see every decision. Change the `spr_id`s, categories and texts below.

In [31]:
one = q("SELECT spr_id FROM spr_mapping_result WHERE match_status = 'mapped' AND street_match_method = 'extra_words' ORDER BY random() LIMIT 1").spr_id[0]
explain(int(one))

== SPR row 218649


,spr_id,match_type,cad_candidate_ids,region,community,settlement,street,house,house_type,apartment
0,218649,None,None,ԱՐՄԱՎԻՐ,ՎԱՂԱՐՇԱՊԱՏ,ԷՋՄԻԱԾԻՆ,ԲԱՂՐԱՄՅԱՆ Փ.,16,Շ,59



== Step 3 geography: rule=as_is  status=ok  note=None  hint=None
   keys: region='արմավիր' community='վաղարշապատ' settlement='էջմիածին'
   search areas: [['settlement', ['արմավիր', 'վաղարշապատ', 'էջմիածին']]]
   house_k='16' apartment_k='59' house_type='Շ'

== How the street text is read:
   as_written             բաղրամյան | փողոց                             leftover: none 

== Step 4 street: found  rule=extra_words  reading=as_written  searched_in=settlement  type_check=same type  score=nan


,street_uid,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,11337,Էջմիածին,Մարշալ Բաղրամյան,փողոց,None,None,677



== Step 5 address: mapped  pick=single  object_type=same
   cadastre rows with this house on that street: 83


,cadastre_id,house,house_type,building,building_type
0,416492,16,շենք,80,բնակարան
1,414771,16,շենք,26,բնակարան
2,416141,16,շենք,35,բնակարան
3,414090,16,շենք,39,բնակարան
4,416484,16,շենք,69,բնակարան
...,...,...,...,...,...
78,416488,16,շենք,65,բնակարան
79,416490,16,շենք,62,բնակարան
80,416491,16,շենք,64,բնակարան
81,416493,16,շենք,61,բնակարան



== Final


,cadastre_id,match_status,match_confidence,issue_category,issue_owner,issue_detail,suggested_cadastre_id
0,416829,mapped,medium,None,None,None,None


In [32]:
one = q("SELECT spr_id FROM spr_mapping_result WHERE issue_category = 'STREET_AMBIGUOUS' ORDER BY random() LIMIT 1").spr_id[0]
explain(int(one))

== SPR row 156469


,spr_id,match_type,cad_candidate_ids,region,community,settlement,street,house,house_type,apartment
0,156469,None,None,ԱՐԱԳԱԾՈՏՆ,ԱՇՏԱՐԱԿ,ԱՇՏԱՐԱԿ,ԳԻՏԱՎԱՆ,9,Շ,Կ.5



== Step 3 geography: rule=as_is  status=ok  note=None  hint=None
   keys: region='արագածոտն' community='աշտարակ' settlement='աշտարակ'
   search areas: [['settlement', ['արագածոտն', 'աշտարակ', 'աշտարակ']]]
   house_k='9' apartment_k='Կ5' house_type='Շ'

== How the street text is read:
   as_written             գիտավան                                       leftover: none 

== Step 4 street: ambiguous  rule=extra_words  reading=as_written  searched_in=settlement  type_check=issue has no type  score=nan


,street_uid,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,463,Աշտարակ,ՌՖԷԻ գիտավան,թաղամաս,None,None,232
1,506,Աշտարակ,ՖՀԻ գիտավան,թաղամաս,None,None,173



== Final


,cadastre_id,match_status,match_confidence,issue_category,issue_owner,issue_detail,suggested_cadastre_id
0,None,ambiguous,None,STREET_AMBIGUOUS,REVIEW,candidates (extra_words): Աշտարակ: ՌՖԷԻ գիտավան թաղամաս ; Աշտարակ:...,None


In [33]:
sample('OBJECT_TYPE_CONFLICT', 10)

,spr_id,region,community,settlement,street,house,house_type,apartment,cadastre_id,match_status,match_confidence,issue_category,issue_detail,street_match_method
0,418628,ԼՈՌԻ,ԱԼԱՎԵՐԴԻ,ԱԼԱՎԵՐԴԻ,ԱՆԴՐԱՆԻԿԻ Փ.,24,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 539926,extra_words
1,290539,ԵՐԵՎԱՆ,ԷՐԵԲՈՒՆԻ,NaN,ԱՐԻՆ ԲԵՐԴ 5 ՆՐԲ.,10/3,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 1075805,word_form
2,42464,ԵՐԵՎԱՆ,ՆՈՐ ՆՈՐՔ,NaN,Դ. ՄԱԼՅԱՆ Փ.,6,Շ,10,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Շ vs cadastre object of id 966584,exact
3,282324,ԵՐԵՎԱՆ,ԱՐԱԲԿԻՐ,NaN,ՀՐ. ՔՈՉԱՐԻ Փ.,145,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 740416,initial_for_name
4,236041,ԳԵՂԱՐՔՈՒՆԻՔ,ԳԱՎԱՌ,ԿԱՐՄԻՐԳՅՈՒՂ,ՍԵՐԳՈ ԱՎԵՏԻՍՅԱՆ Փ. 1 ՆՐԲ.,1/3,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 239765,initial_for_name
5,57170,ՏԱՎՈՒՇ,ԴԻԼԻՋԱՆ,ԳՈՇ,Մ. ԳՈՇԻ 4 ՆՐԲ.,2,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 217464,exact
6,318261,ԵՐԵՎԱՆ,ՄԱԼԱԹԻԱ-ՍԵԲԱՍՏԻԱ,NaN,ԱՆԴՐԱՆԻԿԻ Փ.,61,Շ,6,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Շ vs cadastre object of id 1137628,extra_words
7,30738,ԵՐԵՎԱՆ,ՆՈՐ ՆՈՐՔ,NaN,Դ. ՄԱԼՅԱՆ Փ.,6,Շ,12,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Շ vs cadastre object of id 966572,exact
8,483117,ԿՈՏԱՅՔ,ԱԲՈՎՅԱՆ,ԱԲՈՎՅԱՆ,7 ՄԿՐՇ. 1 ԹՂՄ.,116,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 454999,exact
9,265890,ԵՐԵՎԱՆ,ԱՋԱՓՆՅԱԿ,NaN,ՖՈՒՉԻԿԻ Փ.,19/11,Տ,NaN,None,suggested,None,OBJECT_TYPE_CONFLICT,SPR house type Տ vs cadastre object of id 714201,extra_words


In [34]:
search_cadastre('բաղրամյան', 'Արաբկիր')

,street_uid,region,community,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,17220,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,2-րդ,նրբանցք,67
1,17221,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,2-րդ,փակուղի,146
2,17222,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,3-րդ,նրբանցք,566
3,17223,Երևան,Երևան,Արաբկիր,Մարշալ Բաղրամյան,պողոտա,NaN,NaN,1922


In [35]:
streets_in('Ծովազարդ').head(30)

,street_uid,settlement,street_name,street_type,sub_street_name,sub_street_type,n_addresses
0,12880,Ծովազարդ,Ա. Բակունցի,փողոց,NaN,NaN,18
1,12881,Ծովազարդ,Ա. Իսահակյան,փողոց,1-ին,փակուղի,1
2,12882,Ծովազարդ,Ա. Իսահակյան,փողոց,2-րդ,փակուղի,1
3,12883,Ծովազարդ,Ա. Իսահակյան,փողոց,NaN,NaN,18
4,12884,Ծովազարդ,Աղբյուր Սերոբի,փողոց,NaN,NaN,23
5,12885,Ծովազարդ,Արաբոյի,փողոց,NaN,NaN,11
6,12886,Ծովազարդ,Գ. Նժդեհի,փողոց,NaN,NaN,7
7,12887,Ծովազարդ,Գ. Չաուշի,փողոց,NaN,NaN,30
8,12890,Ծովազարդ,Զորավար Անդրանիկի,փողոց,1-ին,նրբանցք,8
9,12891,Ծովազարդ,Զորավար Անդրանիկի,փողոց,1-ին,փակուղի,2


Output files are in `output/`:
`spr_mapping_result.csv` (every row), `00_summary.xlsx`, `issues_for_cadastre.xlsx`, `issues_for_spr.xlsx`, `needs_review.xlsx`, `spr_with_id.csv`.

In [36]:
import pandas as pd
df = pd.read_csv("output/spr_mapping_result.csv", dtype=str, encoding="utf-8-sig")
f = df[(df["street_match_method"] == "fuzzy")].copy()
f["similarity_score"] = f["similarity_score"].astype(float)   # so sorting is numeric
fuzzy_streets = (
   f.groupby([ "street","cad_street_name"],
             dropna=False)
    .agg(spr_rows=("spr_id", "size"), example_spr_id=("spr_id", "first"))
    .reset_index()
)
print(len(fuzzy_streets), "distinct street pairs")   # 134
fuzzy_streets.to_csv("output/unique_fuzzy_output.csv", index=False, encoding="utf-8-sig")


630 distinct street pairs


In [37]:
fuzzy_mapped = q("""
SELECT -- r.spr_id, r.region, r.community, r.settlement,
       r.street            AS spr_street,
       -- r.house, r.apartment,
       c.street_name       AS cad_street
       -- c.street_type       AS cad_type,
       -- c.sub_street_name   AS cad_sub_street,
       -- c.house             AS cad_house,
       -- c.building          AS cad_building,
       -- r.similarity_score,
       -- r.text_reading,
       -- r.cadastre_id
FROM spr_mapping_result r
JOIN cadastre c ON c.id = r.cadastre_id
WHERE r.match_status = 'mapped'
  AND r.street_match_method = 'fuzzy'
ORDER BY r.similarity_score, r.region, r.settlement, r.street
""")
print(len(fuzzy_mapped), "rows")
fuzzy_mapped.head(50)

27 rows


,spr_street,cad_street
0,ԴՊՐՈՑԱԿԱՆՆԵՐԻ Փ.,Դպրոցական
1,ՄԻԱՍՆԻԿՅԱՆ Փ.,Ա. Մյասնիկյան
2,Վ. ՄԱՆԻԿՈՆՅԱՆ Փ.,Վ. Մամիկոնյան
3,ԹՈՐՈՄԱՆՅԱՆ Փ.,Թ. Թորամանյան
4,ՆԱԼԲԱՆԴԱՅՆ Փ. 2 ՓԿՂ.,Մ. Նալբանդյան
5,ՍԶԱՏՈՒԹՅԱՆ Փ. 7 ՆՐԲ.,Ազատության
6,ՋՐԱՇԱՆԱՐԱՇՆԵՐԻ ԲԱՆԱՎԱՆ 2 Փ.,Ջրաշինարարների բանավանի 2-րդ
7,ԲԱՆԱՎԱՆ ԱՅԳԵԳՈՐԾԱԿԱՆ ԹՂՄ. 1 Փ.,Գառնի բանավան այգեգործական թաղամասի 1-ին
8,Խ. ԲՈՎՅԱՆ Փ.,Խ. Աբովյան
9,ՍԱՐՈՅԱՆ Փ. 1 ՓԿՂ.,Մ. Սարյան
